In [ ]:
# === BOOTSTRAP: RUN THIS FIRST ===
import os, sys
from pathlib import Path
REPO_PATH = Path("/content/drive/MyDrive/MaintainAI/code")
os.chdir(REPO_PATH)
sys.path.insert(0, str(REPO_PATH))
print("ROOT:", REPO_PATH)
print("Exists:", REPO_PATH.exists())
print("src exists:", (REPO_PATH / "src").exists())

# 07 — QLoRA fine-tuning (Colab GPU)

## Prerequisites
1. Run `01_environment_check` first
2. Mount Google Drive
3. Run `06_base_slm_evaluation` to establish baseline
4. Ensure `data/slm/train.jsonl` and `data/slm/val.jsonl` exist (from notebook 05)

## Method
- 4-bit NF4 base (`Qwen/Qwen2.5-3B-Instruct`) + LoRA adapters
- LoRA: r=16, alpha=32, dropout 0.05, targets q/k/v/o + gate/up/down
- PEFT + TRL SFTTrainer
- Full 3B fine-tuning explicitly NOT attempted (resource audit)
- Config mirrored in `configs/training.yaml`

## Checkpoints
- Drive: `MaintainAI/slm/checkpoints/exp_001/`
- Select checkpoint on VAL loss (never train loss)
- Final adapter → Drive + optionally Hugging Face Hub (LoRA only, ~50-100MB)

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Install dependencies
!pip install -q transformers accelerate bitsandbytes peft trl datasets 2>&1 | tail -1

# Verify GPU
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), 'GB')

In [ ]:
# SMOKE TEST FIRST (20 examples, 10 steps) — verify pipeline works
SMOKE = True  # Set False for full run after smoke passes

import json
from src.train_slm import train

result = train(
    config_path='configs/training.yaml',
    smoke=SMOKE,
    allow_cpu=False  # Requires GPU
)
print(json.dumps(result, indent=1))

In [ ]:
# If smoke test passes, run FULL training (set SMOKE = False above and re-run)
# Full config from configs/training.yaml:
# - epochs: 3
# - batch: 1, grad_accum: 4
# - lr: 2e-4, warmup: 20
# - max_seq_length: 1024
# - fp16 on T4
# - eval/save every 50 steps
# - load_best_model_at_end=True (select on VAL loss)

# Expected runtime on T4: ~30-45 minutes for 640 train + 140 val examples

In [ ]:
# After full training, adapter is at:
# /content/drive/MyDrive/MaintainAI/slm/checkpoints/exp_001/final/

# Optionally push to Hugging Face Hub (LoRA only)
# from huggingface_hub import HfApi
# api = HfApi()
# api.upload_folder(
#     folder_path='/content/drive/MyDrive/MaintainAI/slm/checkpoints/exp_001/final',
#     repo_id='your-username/maintainai-qwen2.5-3b-lora',
#     repo_type='model'
# )
# print('Pushed to Hub')

In [ ]:
# Verify adapter can be loaded
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

ADAPTER_PATH = '/content/drive/MyDrive/MaintainAI/slm/checkpoints/exp_001/final'
BASE_MODEL = 'Qwen/Qwen2.5-3B-Instruct'

tok = AutoTokenizer.from_pretrained(ADAPTER_PATH, trust_remote_code=False)
base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, device_map='auto', trust_remote_code=False)
model = PeftModel.from_pretrained(base, ADAPTER_PATH)
print('✓ Adapter loads successfully')
print(f'Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')